## Deduplicate, validate, flag price-limit-hit days

In [0]:
%run ../config/config

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import BooleanType

%md
#### 1. Read bronze Data

In [0]:
bronze_df = spark.table(tables["raw_ohlcv"]).cache()
bronze_count = bronze_df.count()
print(f"Bronze rows: {bronze_count:,}")
print(f"Bronze symbols: {bronze_df.select('symbol').distinct().count()}")

#### 2. Deduplication

In [0]:
deduped_df = bronze_df.dropDuplicates(["symbol", "date"])

dup_count = bronze_count - deduped_df.count()
print(f"Removed {dup_count:,} duplicate rows")

#### 3. OHLCV validation

In [0]:
validated_df = deduped_df.filter(
    (F.col("open") > 0) &
    (F.col("high") > 0) &
    (F.col("low") > 0) &
    (F.col("close") > 0) &
    (F.col("volume") >= 0) &
    (F.col("high") >= F.col("low")) &
    (F.col("high") >= F.col("open")) &
    (F.col("high") >= F.col("close")) &
    (F.col("low") <= F.col("open")) &
    (F.col("low") <= F.col("close"))
)

invalid_count = deduped_df.count() - validated_df.count()
print(f"Removed {invalid_count:,} wrong OHLCV format")

#### 4. Forward-fill missing values

In [0]:
# Define window 
window_symbol = Window.partitionBy("symbol").orderBy("date")

# Forward fill
for col_name in ohlcv_columns:
    validated_df = validated_df.withColumn(
        col_name,
        F.coalesce(
            F.col(col_name),
            F.last(F.col(col_name), ignorenulls=True).over(window_symbol)
        )
    )

# Drop remaining nulls
filled_df = validated_df.dropna(subset=ohlcv_columns[:4])  # open, high, low, close
print(f"Rows after forward-fill and drop nulls: {filled_df.count():,}")

#### 5. Derived columns

In [0]:
# Previous close
enriched_df = filled_df.withColumn(
    "prev_close", F.lag("close", 1).over(window_symbol)
)

# Daily return: (close - prev_close) / prev_close
enriched_df = enriched_df.withColumn(
    "daily_return",
    F.round(
        F.when(F.col("prev_close").isNotNull() & (F.col("prev_close") > 0),
               (F.col("close") - F.col("prev_close")) / F.col("prev_close"))
        .otherwise(None),
        4
    )
)

# Log return: ln(close / prev_close)
enriched_df = enriched_df.withColumn(
    "log_return",
    F.round(
        F.when(F.col("prev_close").isNotNull() & (F.col("prev_close") > 0),
               F.log(F.col("close") / F.col("prev_close")))
        .otherwise(None),
        4
    )
)

# Typical price: (high + low + close) / 3
enriched_df = enriched_df.withColumn(
    "typical_price",
    F.round((F.col("high") + F.col("low") + F.col("close")) / 3.0, 2)
)

# Spread: high - low
enriched_df = enriched_df.withColumn(
    "spread",
    F.round(F.col("high") - F.col("low"), 2)
)


#### 6. Flag price limit hit days

In [0]:
limit_expr = F.lit(0.0)
for ex, limit in price_limits.items():
    limit_expr = F.when(F.col("exchange") == ex.value, F.lit(limit)).otherwise(limit_expr)

# A day hits price limit if |daily_return| so (exchange_limit - small tolerance).
# Null daily_return: false by definition.
enriched_df = enriched_df.withColumn(
    "price_limit_hit",
    F.when(F.col("daily_return").isNull(), F.lit(False))
    .otherwise(F.abs(F.col("daily_return")) >= (limit_expr - 0.005))
)

#### 7. Select final columns & write

In [0]:
final_columns = [
    "symbol", "date", "open", "high", "low", "close", "volume","exchange", "daily_return", "log_return","typical_price", "spread", "prev_close", "price_limit_hit",
]

silver_df = enriched_df.select(*final_columns)

silver_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tables["cleaned_ohlcv"])
bronze_df.unpersist()

print(f"Successfull writen to {tables['cleaned_ohlcv']}")

#### 8. Verify

In [0]:
%sql
SELECT
    COUNT(*)                         AS total_rows,
    COUNT(DISTINCT symbol)           AS unique_symbols,
    SUM(CASE WHEN close IS NULL THEN 1 ELSE 0 END)        AS null_close,
    SUM(CASE WHEN volume IS NULL THEN 1 ELSE 0 END)       AS null_volume,
    SUM(CASE WHEN daily_return IS NULL THEN 1 ELSE 0 END) AS null_daily_return
FROM wyckoff_catalog.silver.cleaned_ohlcv;

In [0]:
%sql
SELECT *
FROM wyckoff_catalog.silver.cleaned_ohlcv
WHERE symbol = 'HPG'
ORDER BY date
LIMIT 10;